In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
simulator = AerSimulator()

In [4]:
# Helper Functions

def quantum_random_bit():
    """Generate one random bit by measuring |+> = (|0> + |1>) / sqrt(2)."""

    # Create quantum circuit with 1 quibit and 1 classical bit
    qc = QuantumCircuit(1, 1)

    # Apply a Hadamard gate to put |0> into superposition
    # |+> = (|0> + |1>) / sqrt(2)
    qc.h(0)

    # Measure the qubit
    # result is randomly 0 or 1 as qubit is in superposition
    qc.measure(0, 0)

    # Run the circuit once to retriee one random bit
    result = simulator.run(qc, shots=1, memory=True).result()

    # Extract the measured bit and convert it from string to integer
    return int(result.get_memory()[0])


def quantum_random_bits(n):
    """Generate n quantum random bits without using Python's random module."""

    # Generate n seprate quantum random bits by repetedly measuring |+>
    return [quantum_random_bit() for _ in range(n)]


def prepare_qubit(bit, basis):
    """
    Alice prepares one qubit.

    basis = 0: computational / rectilinear basis
        bit 0 -> |0>
        bit 1 -> |1>

    basis = 1: diagonal basis
        bit 0 -> |+>
        bit 1 -> |->
    """

    # Start with a new qubit in the deafult state |0>
    qc = QuantumCircuit(1, 1)

    # If the bit is 1, apply an X gate to change |0> to |1>
    # If the bit is 0, leave the qubit as |0>
    if bit == 1:
        qc.x(0)

    # If the basis is diagonal, apply a Hadamard gate
    # |0> becomes |+>
    # |1> becomes |->
    if basis == 1:
        qc.h(0)

    # Return the prepared qubit circuit
    return qc


def measure_qubit(qubit_circuit, basis):
    """
    Measure one qubit in the chosen basis.

    basis = 0: computational / rectilinear basis
    basis = 1: diagonal basis
    """

    # Copy the circuit so the original qubit circuit is not modified
    qc = qubit_circuit.copy()

    #  If measuring in the diagonal basis, apply H first
    # This converts |+> to |0> and |-> to |1>,
    # so a normal computational measurement can be used.
    if basis == 1:
        qc.h(0)

    # Measure the qubit into the classical bit
    qc.measure(0, 0)

    # Run the measurement once, because each qubit is measured only once
    result = simulator.run(qc, shots=1, memory=True).result()

    # Return the measured bit as an integer
    return int(result.get_memory()[0])


def bit_string(bits):
    """Format a list of bits as a compact string."""

    # Convert a list such as [1, 0, 1, 1] into the string "1011"
    return ''.join(str(bit) for bit in bits)

def resend_qubit(measured_bit, measured_basis):
    """
    Eve sends Bob a fresh qubit based on her measurement result.
    This is the intercept-resend attack.
    """

    # Eve cannot forward the original qubit after measuring it,
    # because measurement may disturb the state.
    # Instead, she prepares a new qubit using the bit and basis she measured.
    return prepare_qubit(measured_bit, measured_basis)

In [5]:
# Run the Protocol

def run_bb84_attacker(num_qubits=100, sample_size=20, threshold=0.20):
    # Print a heading so it is clear this version includes Eve
    print("BB84 with intercept-resend attacker")
    print("=" * 50)

    # Alice

    # Alice generates her random secret bits using quantum randomness
    alice_bits = quantum_random_bits(num_qubits)

    # Alice randomly chooses a basis for each bit:
    # 0 = computational basis, 1 = diagonal basis
    alice_bases = quantum_random_bits(num_qubits)

    # Alice prepares one qubit for each bit and basis choice
    qubits = [prepare_qubit(bit, basis) for bit, basis in zip(alice_bits, alice_bases)]

    # Confirm that Alice has prepared the qubits for sending
    print("Alice prepared", num_qubits, "qubits.")

    # Eve

    # Eve randomly chooses a measurement basis for each intercepted qubit
    # She does not know Alice's original bases
    eve_bases = quantum_random_bits(num_qubits)

    # This list stores the bits Eve obtains from her measurements
    eve_bits = []

    # This list stores the new qubits Eve prepares and sends to Bob
    resent_qubits = []

    # Eve intercepts every qubit sent by Alice
    for qubit, eve_basis in zip(qubits, eve_bases):

        # Eve measures the qubit using her randomly chosen basis
        eve_bit = measure_qubit(qubit, eve_basis)

        # Eve records her measurement result as her guess of Alice's bit
        eve_bits.append(eve_bit)

        # Eve prepares a new qubit using her measured bit and basis
        # This replacement qubit is sent on to Bob
        resent_qubits.append(resend_qubit(eve_bit, eve_basis))

    # Bob receives Eve's resent qubits instead of Alice's original qubits
    qubits = resent_qubits

    # Confirm that Eve has carried out the intercept-resend attack
    print("Eve intercepted, measured, and resent all qubits.")

    # Bob

    # Bob randomly chooses which basis to use for each measurement
    bob_bases = quantum_random_bits(num_qubits)

    # Bob measures each qubit he receives using his chosen basis
    bob_bits = [measure_qubit(qubit, basis) for qubit, basis in zip(qubits, bob_bases)]

    # Confirm that Bob has measured all received qubits
    print("Bob measured the qubits.")

    # Public basis comparison and key sifting

    # Alice and Bob publicly compare only their bases, not their bit values.
    # They keep positions where Alice and Bob used the same basis.
    matching_indices = [i for i in range(num_qubits) if alice_bases[i] == bob_bases[i]]

    # Alice's sifted key contains her bits where her basis matched Bob's
    alice_sifted_key = [alice_bits[i] for i in matching_indices]

    # Bob's sifted key contains his measured bits at the same positions
    bob_sifted_key = [bob_bits[i] for i in matching_indices]

    # Eve's corresponding guesses are shown to demonstrate what she learned
    eve_sifted_guess = [eve_bits[i] for i in matching_indices]

    # Display how many bits remain after basis comparison
    print("Number of matching Alice/Bob bases:", len(matching_indices))

    # Display Alice's and Bob's sifted keys so any errors can be seen
    print("Alice sifted key:", bit_string(alice_sifted_key))
    print("Bob sifted key:  ", bit_string(bob_sifted_key))

    # Display Eve's guesses to show that she tried to learn the key
    print("Eve's guesses:   ", bit_string(eve_sifted_guess))

    # Test part of the sifted key for evidence of attack

    # If there are not enough matching bits, the selected test size is too large
    if len(alice_sifted_key) < sample_size:
        print("Not enough matching bits for the chosen sample size.")
        return [], []

    # Alice publicly reveals the first part of her sifted key for testing
    alice_test = alice_sifted_key[:sample_size]

    # Bob reveals the corresponding part of his sifted key
    bob_test = bob_sifted_key[:sample_size]

    # Count how many revealed test bits are different
    errors = sum(1 for a, b in zip(alice_test, bob_test) if a != b)

    # Calculate the proportion of tested bits that do not match
    error_rate = errors / sample_size

    # Display the attack-detection test results
    print("Testing first", sample_size, "sifted bits.")
    print("Errors:", errors)
    print("Error rate:", round(error_rate, 3))
    print("Detection threshold:", threshold)

    # If the error rate is above the threshold, Eve's disturbance is detected
    if error_rate > threshold:
        print("Attack detected. Key discarded.")
        return [], []

    # Final key

    # Remove the tested bits because they were publicly revealed
    final_alice_key = alice_sifted_key[sample_size:]
    final_bob_key = bob_sifted_key[sample_size:]

    # If the error rate is not above the threshold, the key is accepted for this run
    print("No attack detected in this run. Key accepted.")

    # Display the final keys after removing the test bits
    print("Final Alice key:", bit_string(final_alice_key))
    print("Final Bob key:  ", bit_string(final_bob_key))

    # Return the final keys so they can be compared or used later
    return final_alice_key, final_bob_key

In [ ]:
final_alice_key, final_bob_key = run_bb84_attacker(
    num_qubits=100,
    sample_size=20,
    threshold=0.20
)
